In [ ]:
import os, subprocess, sys
if not os.path.exists(os.path.join('src', 'ns5_core.py')):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Yukinoshita-lin/nsf5-steganography.git', '.'], check=True)
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
for mod, pkg in [('numpy', 'numpy'), ('PIL', 'Pillow'),
                 ('matplotlib', 'matplotlib'), ('sklearn', 'scikit-learn'),
                 ('joblib', 'joblib'), ('pandas', 'pandas')]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import numpy as np
print('project ready:', os.path.exists('src/ns5_core.py'))

In [ ]:
import numpy as np
from ns5_core import embed_string
from fsfeatures import get_lib

def make_pairs(n=6):
    clean, stego = [], []
    for i in range(n):
        y = np.linspace(0, 200, 512)[None, :]
        x = np.linspace(0, 120 + i * 8, 512)[:, None]
        c = np.clip(90 + x + y + np.random.default_rng(i).integers(-8, 9, (512, 512)), 0, 255).astype(np.uint8)
        s, _, _ = embed_string(c, 'S' * 3000, method='nsF5', p=3)
        clean.append(c); stego.append(s)
    return clean, stego
clean, stego = make_pairs()
print('samples:', len(clean), 'clean +', len(stego), 'stego')

In [ ]:
# 纯 Python 11 维特征（无需 Windows DLL，可在 Colab/Linux 运行）
from py_features import features as py_features
FEAT = ['Rm', 'Sm', 'Rn', 'Sn', 'RS_Gr', 'RS_Gn', 'chi2_pvalue',
       'diff_entropy', 'lsb_diff_entropy', 'median_prefix_p', 'chi2_stat']
def feats(im):
    f = py_features(im)
    return [f[k] for k in FEAT]
X = np.vstack([feats(im) for im in clean + stego])
y = np.array([0] * len(clean) + [1] * len(stego))
print('X', X.shape, 'labels', y.sum(), 'stego /', len(y))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
clf = LogisticRegression(max_iter=2000)
auc = cross_val_score(clf, X, y, cv=3, scoring='roc_auc')
print('3-fold AUC (illustrative demo): %.3f' % np.mean(auc))

In [ ]:
import sys, os
sys.path.insert(0, 'src')
from ml_predict import MLPredictor
pred_143 = MLPredictor()
print('143d available:', pred_143.available)
if pred_143.available:
    print('clean prob: %.3f' % pred_143.predict(clean[0])['probability'])
    print('stego prob: %.3f' % pred_143.predict(stego[0])['probability'])

In [ ]:
pred_53 = MLPredictor(
    model_path='models/stego_classifier_v2_jpeg_lgb_51d.joblib',
    clip_outliers=False)
if pred_53.available:
    r53 = pred_53.predict(stego[0])
    print('53d available:', True, 'stego prob:', r53['probability'])
    print('53d verdict:', r53['verdict'])